In [19]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

# treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    use_sud=False,
    matrix_type="coverage",
    # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
        excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own", r"Degree=Pos"]
)

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(corpus.feature_matrix)

In [25]:
import numpy as np
results = []
lex_units_with_different_neighbours = set()
for idx, lex_unit in corpus._idx2lexunit.items():
    current_category = lex_unit[1]
    closest_neighbours = np.argsort(similarity_matrix[idx])[:-21:-1]
    closest_neighbours = [i for i in closest_neighbours if i != idx][:20]
    for neighbour_idx in closest_neighbours:
        neighbour_category = corpus._idx2lexunit[neighbour_idx][1]
        if neighbour_category != current_category and similarity_matrix[idx][neighbour_idx] > 0.6:
            diff = np.abs(corpus.feature_matrix[idx] - corpus.feature_matrix[neighbour_idx])
            closest_features_idx= np.argsort(diff)[:-6:-1]
            closest_features = [corpus._idx2feature[i] for i in closest_features_idx]
                
            results.append({
                "lex_unit": lex_unit,
                "neighbour_lex_unit": corpus._idx2lexunit[neighbour_idx],
                "distance": similarity_matrix[idx][neighbour_idx],
                "closest_features": closest_features,
                
            })
            lex_units_with_different_neighbours.add(lex_unit)
    

for r in results:
    print(r)   

{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Factor', 'PROPN'), 'distance': 0.8279907818128358, 'closest_features': ['node:X:child:rel_shallow=nummod', 'node:X:child:rel_shallow=flat', 'node:X:prev:upos=ADP', 'node:X:parent:Number=Sing', 'node:X:child:rel_shallow=punct']}
{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Q', 'PROPN'), 'distance': 0.782434503177179, 'closest_features': ['node:X:child:rel_shallow=flat', 'node:X:child:rel_shallow=nummod', 'node:X:parent:Number=Sing', 'node:X:child:upos=PUNCT', 'node:X:child:rel_shallow=punct']}
{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Wave', 'PROPN'), 'distance': 0.7689568994602162, 'closest_features': ['node:X:child:rel_shallow=flat', 'node:X:child:rel_shallow=nummod', 'node:X:parent:upos=NOUN', 'node:X:parent:position=after', 'node:X:parent:Number=Sing']}
{'lex_unit': ('$', 'SYM'), 'neighbour_lex_unit': ('Figure', 'PROPN'), 'distance': 0.7291987385247293, 'closest_features': ['node:X:child:rel_shallow=flat', 'node:X:chi

In [22]:
len(lex_units_with_different_neighbours)

830

In [23]:
category_counts = {}
for idx, lex_unit in corpus._idx2lexunit.items():
    category = lex_unit[1]
    if category not in category_counts:
        category_counts[category] = {"total": 0, "with_different_neighbour": 0}
    category_counts[category]["total"] += 1
    if lex_unit in lex_units_with_different_neighbours:
        category_counts[category]["with_different_neighbour"] += 1

for category, counts in category_counts.items():
    total = counts["total"]
    with_diff = counts["with_different_neighbour"]
    percentage = (with_diff / total) * 100 if total > 0 else 0
    print(f"Category: {category}, Total: {total}, With Different Neighbour: {with_diff}, Percentage: {percentage:.2f}%")

Category: SYM, Total: 4, With Different Neighbour: 4, Percentage: 100.00%
Category: CCONJ, Total: 8, With Different Neighbour: 8, Percentage: 100.00%
Category: PART, Total: 3, With Different Neighbour: 3, Percentage: 100.00%
Category: ADP, Total: 52, With Different Neighbour: 18, Percentage: 34.62%
Category: NUM, Total: 70, With Different Neighbour: 31, Percentage: 44.29%
Category: NOUN, Total: 856, With Different Neighbour: 188, Percentage: 21.96%
Category: PROPN, Total: 214, With Different Neighbour: 191, Percentage: 89.25%
Category: ADJ, Total: 286, With Different Neighbour: 120, Percentage: 41.96%
Category: PRON, Total: 38, With Different Neighbour: 38, Percentage: 100.00%
Category: VERB, Total: 375, With Different Neighbour: 21, Percentage: 5.60%
Category: DET, Total: 17, With Different Neighbour: 17, Percentage: 100.00%
Category: ADV, Total: 141, With Different Neighbour: 117, Percentage: 82.98%
Category: SCONJ, Total: 29, With Different Neighbour: 29, Percentage: 100.00%
Categor